In [1]:
# This is me starting
# I need to initiatialise my env variables first, since they provide me access
# In theory, as I'm doing things locally, I don't pass anything yet. But as soon as I add Tavily, I will need to add my key. If I add Langsmith, I will need to add the key.
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Initialise the model
# Create the agent to be able to interact with the model and add tools and system prompts
from langchain.agents import create_agent # the function used to create the agent
from langchain.messages import HumanMessage # the message format that the agent receives from a human
from langgraph.checkpoint.memory import InMemorySaver # the method of checkpointing, in this case using my host's memory - ephemeral.
from langchain.tools import tool
from typing import Any
from tavily import TavilyClient
from pydantic import BaseModel

In [3]:
from ipywidgets import FileUpload
from IPython.display import display
 
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [4]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [15]:
class IngredientsList(BaseModel):
    ingredients: list[str]

# define a system prompt to remind the agent about its purpose each time the agent is invoked
ingredients_agent_system_prompt = "You analyse the image provided by the user and return a detailed list of food ingredients."
ingredients_agent = create_agent(
    model="ollama:gemma4",
    system_prompt=ingredients_agent_system_prompt,
    response_format=IngredientsList
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Please list the ingredients in this photo."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

ingredients_agent_response = ingredients_agent.invoke(
    {"messages": [multimodal_question]}
)

print(ingredients_agent_response['messages'][-1].content)

# just the structured_response
ingredients_agent_response["structured_response"]

# just the ingredients
ingredients = ingredients_agent_response["structured_response"].ingredients

Returning structured response: ingredients=['Leafy Greens (e.g., Lettuce, Mixed Greens)', 'Spinach', 'Tomatoes (Red and Yellow)', 'Bell Peppers (Green, Red, and Yellow)', 'Cucumbers', 'Celery', 'Mushrooms', 'Mixed Fresh Vegetables', 'Packaged Deli Meats/Dairy Items', 'Prepared/Packaged Salads']


In [ ]:
### Recipe agent onwards
# TODO: It would be good to have a human-in-the-loop here to check the ingredients list and correct it if necessary.

tavily_client = TavilyClient()

# define the web searching tool, Tavily offers other options beyond search, but we'll keep it as a search for now
@tool
def web_search(query: str) -> dict[str, Any]:
    """Search the web to find recipes based on the user's query
    
    Args:
        query: Search terms to look for on the internet.
    """
    return tavily_client.search(query)

# define a system prompt to remind the agent about its purpose each time the agent is invoked
recipe_agent_system_prompt = "You are a personal chef. You will suggest recipes based on the ingredients provided by the user. Use the web_search tool to look up recipes on the internet."
recipe_agent = create_agent(
    model="ollama:gemma4:e2b",
    system_prompt=recipe_agent_system_prompt,
    checkpointer=InMemorySaver(), # I am fine with the agent losing its memory if it restarts as it's just local. In production, this would use another method of memory management, e.g. PostgresSaver or AsyncPostgresSaver
    tools=[web_search]
)
# help(agent.invoke)

config = {"configurable": {"thread_id": "1"}}
question = HumanMessage(content=f"Can you suggest a soothing recipe for a wet autumn day? I have the following ingredients: {ingredients}")
response = recipe_agent.invoke(
    {"messages": [question]},
    config
)

In [43]:
# ### Recipe agent onwards
# tavily_client = TavilyClient()

# # define the web searching tool, Tavily offers other options beyond search, but we'll keep it as a search for now
# @tool
# def web_search(query: str) -> dict[str, Any]:
#     """Search the web to find recipes based on the user's query
    
#     Args:
#         query: Search terms to look for on the internet.
#     """
#     return tavily_client.search(query)

# # define a system prompt to remind the agent about its purpose each time the agent is invoked
# recipe_agent_system_prompt = "You are a personal chef. You will suggest recipes based on the ingredients provided by the user. Use the web_search tool to look up recipes on the internet."
# recipe_agent = create_agent(
#     model="ollama:gemma4:e2b",
#     system_prompt=recipe_agent_system_prompt,
#     checkpointer=InMemorySaver(), # I am fine with the agent losing its memory if it restarts as it's just local. In production, this would use another method of memory management, e.g. PostgresSaver or AsyncPostgresSaver
#     tools=[web_search]
# )
# # help(agent.invoke)

# config = {"configurable": {"thread_id": "1"}}
# question = HumanMessage(content=f"Can you suggest a soothing recipe for a wet autumn day? I have the following ingredients: {ingredients}")
# response = recipe_agent.invoke(
#     {"messages": [question]},
#     config
# )

In [18]:
print(response["messages"][-1].content)

That sounds like the perfect time for something light, fresh, and hydrating to combat that damp, moody autumn weather. As your personal chef, I suggest a **Mediterranean Garden Salad with Warm Mushrooms and Light Feta**.

This recipe uses a wide variety of your fresh vegetables to create a balanced, soothing meal that is refreshing without being heavy.

### 🌿 Soothing Mediterranean Garden Salad

This salad is packed with hydration from the cucumbers and bell peppers, balanced by the earthy mushrooms and fresh greens.

**Prep time:** 15 minutes
**Serves:** 2

#### 🥗 Ingredients You Have:

*   **Base:** 4 cups Mixed Leafy Greens or Spinach
*   **Color & Crunch:**
    *   1 large Cucumber, sliced
    *   1 Red Bell Pepper, sliced
    *   1 Yellow Bell Pepper, sliced
    *   1/2 Red Tomato, halved
    *   1/2 Yellow Tomato, halved
    *   1/2 cup Mushrooms, sliced (cremini or button work well)
    *   1/2 cup Celery, thinly sliced
*   **Flavor & Finish:**
    *   1/2 cup prepared/packaged 

In [19]:
print(response["messages"])

[HumanMessage(content="Can you suggest a soothing recipe for a wet autumn day? I have the following ingredients: ['Leafy Greens (e.g., Lettuce, Mixed Greens)', 'Spinach', 'Tomatoes (Red and Yellow)', 'Bell Peppers (Green, Red, and Yellow)', 'Cucumbers', 'Celery', 'Mushrooms', 'Mixed Fresh Vegetables', 'Packaged Deli Meats/Dairy Items', 'Prepared/Packaged Salads']", additional_kwargs={}, response_metadata={}, id='2bde3601-83bb-43cb-bd7f-af0e5814f8a9'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-10-10T21:38:52.182797Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16285210166, 'load_duration': 4219539333, 'prompt_eval_count': 206, 'prompt_eval_duration': 344652000, 'eval_count': 450, 'eval_duration': 11718209000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a127c0-e6f8-7272-bc29-be360341097a-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'soothing autumn vegetabl

In [ ]:
# There is something called langsmith studio, which allows you to use  an interface in your browser to prompt your agent and observe its thought process in langsmith.

In [ ]:
# Below is from the module

In [ ]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="ollama:gemma4",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response)